# Premier League Data Scraping Notebook

This notebook combines three scraping scripts:
1. **Team Player Ages** - Scrape player ages from Transfermarkt (2000-2025)
2. **Team Player Values** - Scrape player market values from Transfermarkt (2000-2025)
3. **Latest Match Data** - Scrape latest EPL match data from football-data.co.uk

## Table of Contents
- [Setup and Imports](#setup)
- [Configuration](#config)
- [Part 1: Scrape Team Player Ages](#ages)
- [Part 2: Scrape Team Player Values](#values)
- [Part 3: Scrape Latest Match Data](#matches)
- [Run All Scraping Tasks](#run-all)

## Setup and Imports <a id='setup'></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os
import io
from datetime import datetime
from urllib.parse import urljoin

print("All libraries imported successfully!")

## Configuration <a id='config'></a>

In [ ]:
# Headers to avoid being blocked by websites
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Base URLs
BASE_URL = "https://www.transfermarkt.com"
FOOTBALL_DATA_URL = "https://www.football-data.co.uk/mmz4281/{season}/E0.csv"

# Years to scrape (2000-2025)
YEARS_TO_SCRAPE = list(range(2000, 2026))

# Premier League teams
PREMIER_LEAGUE_TEAMS = {
    'Arsenal': '/arsenal-fc/startseite/verein/11',
    'Aston Villa': '/aston-villa/startseite/verein/405',
    'Bournemouth': '/afc-bournemouth/startseite/verein/989',
    'Brentford': '/brentford-fc/startseite/verein/1148',
    'Brighton': '/brighton-amp-hove-albion/startseite/verein/1237',
    'Burnley': '/fc-burnley/startseite/verein/1132',
    'Chelsea': '/fc-chelsea/startseite/verein/631',
    'Crystal Palace': '/crystal-palace/startseite/verein/873',
    'Everton': '/fc-everton/startseite/verein/29',
    'Fulham': '/fc-fulham/startseite/verein/931',
    'Liverpool': '/fc-liverpool/startseite/verein/31',
    'Leeds United': '/leeds-united/startseite/verein/399',
    'Leicester City': '/leicester-city/startseite/verein/1003',
    'Luton Town': '/luton-town-fc/startseite/verein/1031',
    'Manchester City': '/manchester-city/startseite/verein/281',
    'Manchester United': '/manchester-united/startseite/verein/985',
    'Newcastle': '/newcastle-united/startseite/verein/762',
    'Norwich City': '/norwich-city/startseite/verein/1123',
    'Nottingham Forest': '/nottingham-forest/startseite/verein/703',
    'Sheffield United': '/sheffield-united/startseite/verein/350',
    'Southampton': '/fc-southampton/startseite/verein/180',
    'Tottenham': '/tottenham-hotspur/startseite/verein/148',
    'Watford': '/fc-watford/startseite/verein/1010',
    'West Bromwich': '/west-bromwich-albion/startseite/verein/984',
    'West Ham': '/west-ham-united/startseite/verein/379',
    'Wolverhampton': '/wolverhampton-wanderers/startseite/verein/543'
}

# Output directory
OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration set. Output directory: {OUTPUT_DIR}")
print(f"Number of teams: {len(PREMIER_LEAGUE_TEAMS)}")
print(f"Years to scrape: {min(YEARS_TO_SCRAPE)} - {max(YEARS_TO_SCRAPE)}")

## Part 1: Scrape Team Player Ages <a id='ages'></a>

This section scrapes player ages from Transfermarkt for all Premier League teams from 2000 to 2025.

### Helper Functions for Age Scraping

In [ ]:
def get_team_squad_url(team_url, year=None):
    """Convert team home URL to squad URL with optional year parameter"""
    if year and year < 2025:
        return team_url.replace('/startseite/', f'/kader/verein/{team_url.split("/")[-1]}/saison_id/{year}')
    else:
        return team_url.replace('/startseite/', '/kader/')

def parse_birth_date(date_str, reference_year=None):
    """Parse birth date string and calculate age for a specific year"""
    if not date_str or date_str == '-':
        return None
    
    if not reference_year:
        reference_year = datetime.now().year
    
    try:
        date_formats = ['%b %d, %Y', '%d.%m.%Y', '%Y-%m-%d', '%m/%d/%Y']
        birth_date = None
        for fmt in date_formats:
            try:
                birth_date = datetime.strptime(date_str.strip(), fmt)
                break
            except ValueError:
                continue
        
        if birth_date:
            age = reference_year - birth_date.year
            return age
        else:
            year_match = re.search(r'(\d{4})', date_str)
            if year_match:
                birth_year = int(year_match.group(1))
                return reference_year - birth_year
    except Exception as e:
        print(f"Error parsing date '{date_str}': {str(e)}")
    
    return None

def parse_age_directly(age_str):
    """Parse age if it's directly provided as a number"""
    if not age_str or age_str == '-':
        return None
    try:
        age_match = re.search(r'(\d+)', age_str.strip())
        if age_match:
            return int(age_match.group(1))
    except:
        pass
    return None

print("Age scraping helper functions defined.")

In [ ]:
def scrape_team_player_ages(team_name, team_url, year=None):
    """Scrape player ages for a specific team and year"""
    year_str = f" ({year})" if year else ""
    print(f"Scraping player ages for {team_name}{year_str}...")
    
    if year and year < 2025:
        parts = team_url.strip('/').split('/')
        team_slug = parts[0]
        team_id = parts[-1]
        squad_url = f"{BASE_URL}/{team_slug}/kader/verein/{team_id}/saison_id/{year}"
    else:
        squad_url = BASE_URL + get_team_squad_url(team_url)
    
    try:
        response = requests.get(squad_url, headers=HEADERS)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        player_ages = []
        squad_table = soup.find('table', {'class': 'items'})
        if not squad_table:
            print(f"Could not find squad table for {team_name}{year_str}")
            return []
        
        player_rows = squad_table.find('tbody').find_all('tr')
        
        for row in player_rows:
            if 'thead' in str(row) or not row.find('td'):
                continue
            
            name_cell = row.find('td', {'class': 'hauptlink'})
            if name_cell:
                name_link = name_cell.find('a')
                player_name = name_link.text.strip() if name_link else 'Unknown'
            else:
                continue
            
            age = None
            age_cells = row.find_all('td', {'class': 'zentriert'})
            for cell in age_cells:
                cell_text = cell.text.strip()
                age = parse_age_directly(cell_text)
                if age:
                    break
                age = parse_birth_date(cell_text, year)
                if age:
                    break
            
            if age and 16 <= age <= 45:
                player_ages.append({
                    'player_name': player_name,
                    'age': age
                })
        
        return player_ages
    except Exception as e:
        print(f"Error scraping {team_name}{year_str}: {str(e)}")
        return []

def calculate_age_statistics(player_ages):
    """Calculate total and average age for a team"""
    if not player_ages:
        return 0, 0, 0
    
    ages = [player['age'] for player in player_ages]
    total_age = sum(ages)
    average_age = total_age / len(ages) if ages else 0
    player_count = len(ages)
    
    return total_age, average_age, player_count

print("Age scraping main functions defined.")

### Run Age Scraping

In [ ]:
def scrape_all_team_ages():
    """Scrape player ages for all teams across all years"""
    print("Starting Premier League player age scraping for 2000-2025...")
    all_data = []
    
    for year in YEARS_TO_SCRAPE:
        print(f"\n{'='*50}")
        print(f"SCRAPING DATA FOR YEAR: {year}")
        print(f"{'='*50}")
        
        for team_name, team_url in PREMIER_LEAGUE_TEAMS.items():
            player_ages = scrape_team_player_ages(team_name, team_url, year)
            total_age, avg_age, player_count = calculate_age_statistics(player_ages)
            
            team_data = {
                'year': year,
                'team_name': team_name,
                'total_age_years': total_age,
                'average_age_years': round(avg_age, 2),
                'number_of_players': player_count
            }
            
            all_data.append(team_data)
            print(f"{team_name} ({year}): Total Age = {total_age} years, Average Age = {avg_age:.2f} years, Players = {player_count}")
            time.sleep(3)
    
    df = pd.DataFrame(all_data)
    output_file = os.path.join(OUTPUT_DIR, 'premier_league_team_ages_2000_2025.csv')
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\nMulti-year data saved to {output_file}")
    
    year_summary = df.groupby('year').agg({
        'total_age_years': 'sum',
        'average_age_years': 'mean',
        'number_of_players': 'sum'
    }).round(2)
    
    print(f"\nSummary by year:")
    print(year_summary)
    
    summary_file = os.path.join(OUTPUT_DIR, 'premier_league_ages_yearly_summary.csv')
    year_summary.to_csv(summary_file, encoding='utf-8')
    print(f"Yearly summary saved to {summary_file}")
    
    return df, year_summary

# Uncomment the line below to run age scraping
# df_ages, summary_ages = scrape_all_team_ages()

## Part 2: Scrape Team Player Values <a id='values'></a>

This section scrapes player market values from Transfermarkt for all Premier League teams from 2000 to 2025.

### Helper Functions for Value Scraping

In [ ]:
def parse_market_value(value_str):
    """Parse market value string and convert to millions of euros"""
    if not value_str or value_str == '-':
        return 0
    
    value_str = value_str.replace('€', '').replace('$', '').replace('£', '').strip()
    
    if 'm' in value_str.lower():
        return float(re.sub(r'[^\d.]', '', value_str))
    elif 'k' in value_str.lower():
        return float(re.sub(r'[^\d.]', '', value_str)) / 1000
    else:
        try:
            return float(re.sub(r'[^\d.]', '', value_str))
        except:
            return 0

print("Value scraping helper functions defined.")

In [ ]:
def scrape_team_player_values(team_name, team_url, year=None):
    """Scrape player market values for a specific team and year"""
    year_str = f" ({year})" if year else ""
    print(f"Scraping player values for {team_name}{year_str}...")
    
    if year and year < 2025:
        parts = team_url.strip('/').split('/')
        team_slug = parts[0]
        team_id = parts[-1]
        squad_url = f"{BASE_URL}/{team_slug}/kader/verein/{team_id}/saison_id/{year}"
    else:
        squad_url = BASE_URL + get_team_squad_url(team_url)
    
    try:
        response = requests.get(squad_url, headers=HEADERS)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        player_values = []
        squad_table = soup.find('table', {'class': 'items'})
        if not squad_table:
            print(f"Could not find squad table for {team_name}{year_str}")
            return []
        
        player_rows = squad_table.find('tbody').find_all('tr')
        
        for row in player_rows:
            if 'thead' in str(row) or not row.find('td'):
                continue
            
            name_cell = row.find('td', {'class': 'hauptlink'})
            if name_cell:
                name_link = name_cell.find('a')
                player_name = name_link.text.strip() if name_link else 'Unknown'
            else:
                continue
            
            value_cell = row.find('td', {'class': 'rechts hauptlink'})
            if value_cell:
                value_text = value_cell.text.strip()
                market_value = parse_market_value(value_text)
                player_values.append({
                    'player_name': player_name,
                    'market_value_millions': market_value
                })
        
        return player_values
    except Exception as e:
        print(f"Error scraping {team_name}{year_str}: {str(e)}")
        return []

def calculate_team_statistics(player_values):
    """Calculate total and average market value for a team"""
    if not player_values:
        return 0, 0, 0
    
    values = [player['market_value_millions'] for player in player_values]
    total_value = sum(values)
    average_value = total_value / len(values) if values else 0
    player_count = len(values)
    
    return total_value, average_value, player_count

print("Value scraping main functions defined.")

### Run Value Scraping

In [ ]:
def scrape_all_team_values():
    """Scrape player values for all teams across all years"""
    print("Starting Premier League player value scraping for 2000-2025...")
    all_data = []
    
    for year in YEARS_TO_SCRAPE:
        print(f"\n{'='*50}")
        print(f"SCRAPING DATA FOR YEAR: {year}")
        print(f"{'='*50}")
        
        for team_name, team_url in PREMIER_LEAGUE_TEAMS.items():
            player_values = scrape_team_player_values(team_name, team_url, year)
            total_value, avg_value, player_count = calculate_team_statistics(player_values)
            
            team_data = {
                'year': year,
                'team_name': team_name,
                'total_market_value_millions': round(total_value, 2),
                'average_market_value_millions': round(avg_value, 2),
                'number_of_players': player_count
            }
            
            all_data.append(team_data)
            print(f"{team_name} ({year}): Total = €{total_value:.2f}M, Average = €{avg_value:.2f}M, Players = {player_count}")
            time.sleep(3)
    
    df = pd.DataFrame(all_data)
    output_file = os.path.join(OUTPUT_DIR, 'premier_league_team_values_2000_2025.csv')
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\nMulti-year data saved to {output_file}")
    
    year_summary = df.groupby('year').agg({
        'total_market_value_millions': 'sum',
        'average_market_value_millions': 'mean',
        'number_of_players': 'sum'
    }).round(2)
    
    print(f"\nSummary by year:")
    print(year_summary)
    
    summary_file = os.path.join(OUTPUT_DIR, 'premier_league_values_yearly_summary.csv')
    year_summary.to_csv(summary_file, encoding='utf-8')
    print(f"Yearly summary saved to {summary_file}")
    
    return df, year_summary

# Uncomment the line below to run value scraping
# df_values, summary_values = scrape_all_team_values()

## Part 3: Scrape Latest Match Data <a id='matches'></a>

This section scrapes the latest EPL match data from football-data.co.uk (May 2025 - November 2025).

### Helper Functions for Match Data Scraping

In [ ]:
def get_season_string(year):
    """
    Convert year to season string format (e.g., 2425 for 2024-2025 season)
    """
    if year == 2025:
        return "2425"
    elif year == 2026:
        return "2526"
    return f"{str(year)[-2:]}{str(year+1)[-2:]}"

def fetch_season_data(season_string):
    """Fetch Premier League data for a specific season from football-data.co.uk"""
    url = FOOTBALL_DATA_URL.format(season=season_string)
    print(f"Fetching data from: {url}")
    
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        df = pd.read_csv(io.StringIO(response.text))
        print(f"Successfully fetched {len(df)} matches for season {season_string}")
        return df
    except Exception as e:
        print(f"Error fetching data for season {season_string}: {str(e)}")
        return None

def filter_date_range(df, start_date, end_date):
    """Filter dataframe to only include matches between start_date and end_date"""
    if df is None or df.empty:
        return pd.DataFrame()
    
    date_formats = ['%d/%m/%Y', '%d/%m/%y', '%Y-%m-%d']
    
    for fmt in date_formats:
        try:
            df['Date_parsed'] = pd.to_datetime(df['Date'], format=fmt)
            break
        except:
            continue
    
    if 'Date_parsed' not in df.columns:
        print("Warning: Could not parse dates, returning all data")
        return df
    
    mask = (df['Date_parsed'] >= start_date) & (df['Date_parsed'] <= end_date)
    filtered_df = df[mask].copy()
    filtered_df = filtered_df.drop('Date_parsed', axis=1)
    
    print(f"Filtered to {len(filtered_df)} matches between {start_date} and {end_date}")
    return filtered_df

print("Match data scraping helper functions defined.")

In [ ]:
def standardize_column_names(df):
    """Ensure column names match the format in epl-training.csv"""
    required_columns = [
        'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 
        'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 
        'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
    ]
    
    existing_columns = [col for col in required_columns if col in df.columns]
    missing_columns = [col for col in required_columns if col not in df.columns]
    
    if missing_columns:
        print(f"Warning: Missing columns: {missing_columns}")
        for col in missing_columns:
            df[col] = ''
    
    df_standardized = df[required_columns].copy()
    return df_standardized

def standardize_team_names(df):
    """Standardize team names to match the format in epl-training.csv"""
    team_name_mapping = {
        'Man United': 'Man United',
        'Manchester United': 'Man United',
        'Man City': 'Man City',
        'Manchester City': 'Man City',
        'Tottenham': 'Tottenham',
        'Spurs': 'Tottenham',
        'Newcastle': 'Newcastle',
        'Newcastle United': 'Newcastle',
        'West Ham': 'West Ham',
        'West Ham United': 'West Ham',
        'Wolves': 'Wolves',
        'Wolverhampton': 'Wolves',
        "Nott'm Forest": "Nott'm Forest",
        'Nottingham Forest': "Nott'm Forest",
        'Nottingham': "Nott'm Forest",
        'Leicester': 'Leicester',
        'Leicester City': 'Leicester',
        'Brighton': 'Brighton',
        'Brighton & Hove Albion': 'Brighton',
        'Brighton and Hove Albion': 'Brighton',
    }
    
    for col in ['HomeTeam', 'AwayTeam']:
        if col in df.columns:
            df[col] = df[col].replace(team_name_mapping)
    
    return df

print("Match data standardization functions defined.")

### Run Match Data Scraping

In [ ]:
def scrape_latest_match_data():
    """Scrape latest EPL match data (May 2025 - November 2025)"""
    print("Starting Premier League match data scraping (May 2025 - November 2025)...")
    print("="*70)
    
    may_2025_start = datetime(2025, 5, 26)
    may_2025_end = datetime(2025, 5, 31)
    aug_nov_2025_start = datetime(2025, 8, 1)
    aug_nov_2025_end = datetime(2025, 11, 30)
    
    all_matches = []
    
    print("\n--- Fetching 2024-2025 season data (for May 2025) ---")
    df_2425 = fetch_season_data("2425")
    if df_2425 is not None:
        df_may = filter_date_range(df_2425, may_2025_start, may_2025_end)
        if not df_may.empty:
            all_matches.append(df_may)
    
    time.sleep(2)
    
    print("\n--- Fetching 2025-2026 season data (for Aug-Nov 2025) ---")
    df_2526 = fetch_season_data("2526")
    if df_2526 is not None:
        df_aug_nov = filter_date_range(df_2526, aug_nov_2025_start, aug_nov_2025_end)
        if not df_aug_nov.empty:
            all_matches.append(df_aug_nov)
    
    if all_matches:
        combined_df = pd.concat(all_matches, ignore_index=True)
        print(f"\n{'='*70}")
        print(f"Total matches fetched: {len(combined_df)}")
        
        combined_df = standardize_team_names(combined_df)
        combined_df = standardize_column_names(combined_df)
        
        try:
            combined_df['Date_temp'] = pd.to_datetime(combined_df['Date'], format='%d/%m/%Y', errors='coerce')
            combined_df = combined_df.sort_values('Date_temp')
            combined_df = combined_df.drop('Date_temp', axis=1)
        except:
            pass
        
        output_file = os.path.join(OUTPUT_DIR, 'epl-latest-2025.csv')
        combined_df.to_csv(output_file, index=False)
        
        print(f"\nData saved to: {output_file}")
        print("\nSample of fetched data:")
        print(combined_df.head(10))
        
        if not combined_df.empty:
            print(f"\nDate range: {combined_df['Date'].min()} to {combined_df['Date'].max()}")
        
        print("\n" + "="*70)
        print("NEXT STEPS:")
        print("="*70)
        print("1. Review the fetched data in 'data/epl-latest-2025.csv'")
        print("2. If data looks correct, append it to 'data/epl-training.csv'")
        
        return combined_df
    else:
        print("\n⚠ No data was fetched.")
        return None

# Uncomment the line below to run match data scraping
# df_matches = scrape_latest_match_data()

## Run All Scraping Tasks <a id='run-all'></a>

⚠️ **Warning**: Running all scraping tasks will take a very long time (several hours) and make hundreds of requests to external websites. Use with caution and consider running them separately.

In [ ]:
def run_all_scraping():
    """Run all scraping tasks sequentially"""
    print("="*70)
    print("RUNNING ALL SCRAPING TASKS")
    print("="*70)
    print("\nThis will take several hours to complete...\n")
    
    # Task 1: Scrape team ages
    print("\n" + "="*70)
    print("TASK 1: Scraping Team Player Ages")
    print("="*70)
    df_ages, summary_ages = scrape_all_team_ages()
    print("\n✓ Age scraping completed!\n")
    
    # Task 2: Scrape team values
    print("\n" + "="*70)
    print("TASK 2: Scraping Team Player Values")
    print("="*70)
    df_values, summary_values = scrape_all_team_values()
    print("\n✓ Value scraping completed!\n")
    
    # Task 3: Scrape latest match data
    print("\n" + "="*70)
    print("TASK 3: Scraping Latest Match Data")
    print("="*70)
    df_matches = scrape_latest_match_data()
    print("\n✓ Match data scraping completed!\n")
    
    print("\n" + "="*70)
    print("ALL SCRAPING TASKS COMPLETED!")
    print("="*70)
    print(f"\nOutput files saved in: {OUTPUT_DIR}")
    
    return df_ages, summary_ages, df_values, summary_values, df_matches

# Uncomment the line below to run ALL scraping tasks
# df_ages, summary_ages, df_values, summary_values, df_matches = run_all_scraping()

## Usage Instructions

### To run individual scraping tasks:

1. **Age Scraping**: Uncomment and run the cell in [Part 1](#ages) section
   ```python
   df_ages, summary_ages = scrape_all_team_ages()
   ```

2. **Value Scraping**: Uncomment and run the cell in [Part 2](#values) section
   ```python
   df_values, summary_values = scrape_all_team_values()
   ```

3. **Match Data Scraping**: Uncomment and run the cell in [Part 3](#matches) section
   ```python
   df_matches = scrape_latest_match_data()
   ```

### To run all tasks at once:

Uncomment and run the cell in the [Run All](#run-all) section:
```python
df_ages, summary_ages, df_values, summary_values, df_matches = run_all_scraping()
```

### Output Files:

All output files will be saved to the `data` directory:
- `premier_league_team_ages_2000_2025.csv`
- `premier_league_ages_yearly_summary.csv`
- `premier_league_team_values_2000_2025.csv`
- `premier_league_values_yearly_summary.csv`
- `epl-latest-2025.csv`